In [1]:
import pandas as pd

In [2]:


# 1. Load the combined dataset (keeping the original raw dataframe separate)
file_path = '../data/raw/Combined_Employee_Task_Data.csv'
df_raw = pd.read_csv(file_path)

# 2. Create a working copy for preprocessing
df = df_raw.copy()

# 3. Confirm shape and column names
print(f"Working Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

Working Dataset Shape: 11633 rows, 22 columns

Column Names:
01. Timesheet_ID
02. Date
03. Task_ID
04. Task_Name
05. Work_Description
06. Hours_Spent
07. Project_Name
08. Employee_ID
09. Employee_Name
10. Employee_Department
11. Employee_Job_Position
12. Task_Description
13. Timesheet_Work_Logs
14. Original_Task_Description
15. Task_Priority
16. Estimated_Planned_Hours
17. Actual_Hours_Spent
18. Timesheet_Logs_Count
19. Task_Stage
20. Created_Date
21. Deadline_Date
22. All_Collaborating_Employees


In [3]:
# 1. Define columns to drop and document the reasoning
columns_to_drop = [
    'Timesheet_ID',              # Unique identifier for the log; holds no predictive value for modelling.
    'Employee_Name',             # Redundant; perfectly correlates with our target variable 'Employee_ID'.
    'Original_Task_Description'  # Redundant and poor quality; flagged during data prep as having a high null rate, overshadowed by 'Task_Description'.
]

# 2. Drop the redundant columns from our working dataframe
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# 3. Flag columns that carry a high risk of data leakage (do not drop yet, but isolate for review)
# These columns contain information that would not be available at the exact moment a task is assigned to an employee.
leakage_risk_columns = [
    'Actual_Hours_Spent',        # Post-task metric; unknown at assignment time.
    'Task_Stage',                # State of the task currently, which might imply completion rather than assignment state.
    'Timesheet_Logs_Count',      # Aggregated metric calculated after work is logged.
    'Timesheet_Work_Logs',       # Aggregated text from after the work is performed.
    'All_Collaborating_Employees'# Known only after all employees have logged time.
]

print("--- Step 2: Column Removal & Leakage Flagging ---")
print(f"Columns dropped: {columns_to_drop}")
print(f"New dataset shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

print("⚠️ FLAGGED FOR POTENTIAL LEAKAGE (Retained for now):")
for col in leakage_risk_columns:
    print(f" - {col}")

--- Step 2: Column Removal & Leakage Flagging ---
Columns dropped: ['Timesheet_ID', 'Employee_Name', 'Original_Task_Description']
New dataset shape: 11633 rows, 19 columns

⚠️ FLAGGED FOR POTENTIAL LEAKAGE (Retained for now):
 - Actual_Hours_Spent
 - Task_Stage
 - Timesheet_Logs_Count
 - Timesheet_Work_Logs
 - All_Collaborating_Employees


In [4]:
# 1. Check initial missing values
print("--- Missing Values Before Treatment ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n")

--- Missing Values Before Treatment ---
Work_Description           1
Employee_Department       77
Timesheet_Work_Logs       74
Deadline_Date          10012
dtype: int64




In [5]:
# 2. Handle Text Columns
text_columns = ['Work_Description', 'Task_Description', 'Timesheet_Work_Logs', 'All_Collaborating_Employees']
for col in text_columns:
    if col in df.columns:
        # Replacing missing text with an empty string so NLP pipelines (like TF-IDF) don't break
        df[col] = df[col].fillna("")

In [6]:
# 3. Handle Categorical Columns
categorical_columns = ['Project_Name', 'Task_Name', 'Task_Priority', 'Task_Stage', 'Employee_Department', 'Employee_Job_Position']
for col in categorical_columns:
    if col in df.columns:
        # Using "Unknown" for categorical features ensures the model can learn if 'missingness' is a pattern
        df[col] = df[col].fillna("Unknown")

In [7]:
# 4. Handle Numerical Columns
numerical_columns = ['Hours_Spent', 'Estimated_Planned_Hours', 'Actual_Hours_Spent', 'Timesheet_Logs_Count']
for col in numerical_columns:
    if col in df.columns:
        # For timesheet and task metrics, a missing value logically implies 0 (e.g., no planned hours recorded)
        df[col] = df[col].fillna(0.0)

In [8]:
# 5. Handle IDs (Target Variable & Join Keys)
# If Employee_ID (our target) or Task_ID is missing, the row is useless for our specific supervised learning goal.
if df['Employee_ID'].isnull().sum() > 0:
    print(f"Dropping {df['Employee_ID'].isnull().sum()} rows due to missing Employee_ID (Target).")
    df.dropna(subset=['Employee_ID'], inplace=True)

if df['Task_ID'].isnull().sum() > 0:
    print(f"Dropping {df['Task_ID'].isnull().sum()} rows due to missing Task_ID.")
    df.dropna(subset=['Task_ID'], inplace=True)

In [9]:
print("--- Missing Values After Treatment (Excluding Dates) ---")
missing_after = df.drop(columns=['Created_Date', 'Deadline_Date'], errors='ignore').isnull().sum()
print(missing_after[missing_after > 0].to_string() if missing_after.sum() > 0 else "No missing values remaining in processed columns!")

--- Missing Values After Treatment (Excluding Dates) ---
No missing values remaining in processed columns!


In [10]:
# 1. Convert Date Columns to Datetime
date_columns = ['Date', 'Created_Date', 'Deadline_Date']
for col in date_columns:
    if col in df.columns:
        # errors='coerce' turns invalid parsing into NaT (Not a Time) safely
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Convert Metrics to Numeric (Float/Int)
numeric_columns = [
    'Hours_Spent', 
    'Estimated_Planned_Hours', 
    'Actual_Hours_Spent', 
    'Timesheet_Logs_Count'
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Ensure IDs remain categorical identifiers (Strings)
id_columns = ['Task_ID', 'Employee_ID']
for col in id_columns:
    if col in df.columns:
        df[col] = df[col].astype(str)

# 4. Verify the changes
print("--- Step 4: Data Types After Conversion ---")
print(df[date_columns + numeric_columns + id_columns].dtypes)

--- Step 4: Data Types After Conversion ---
Date                       datetime64[us]
Created_Date               datetime64[us]
Deadline_Date              datetime64[us]
Hours_Spent                       float64
Estimated_Planned_Hours           float64
Actual_Hours_Spent                float64
Timesheet_Logs_Count                int64
Task_ID                               str
Employee_ID                           str
dtype: object


In [11]:
print("--- Step 5: Duplicate Handling ---")
initial_rows = df.shape[0]

# 1. Check and remove completely duplicated rows
# This catches accidental double-entries of the exact same timesheet log.
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} completely duplicated rows. Dropping them...")
    df.drop_duplicates(inplace=True)
else:
    print("No completely duplicated rows found.")

# 2. Check for duplicate Timesheet_IDs (if still in the dataframe)
# Timesheet_ID should be strictly unique per log entry.
if 'Timesheet_ID' in df.columns:
    duplicate_ts_ids = df.duplicated(subset=['Timesheet_ID']).sum()
    if duplicate_ts_ids > 0:
        print(f"Found {duplicate_ts_ids} duplicate Timesheet_IDs. Keeping the first occurrence...")
        df.drop_duplicates(subset=['Timesheet_ID'], keep='first', inplace=True)
    else:
        print("No duplicate Timesheet_IDs found.")
else:
    print("Timesheet_ID was removed in Step 2; skipping Timesheet_ID duplicate check.")

# 3. Documenting Task_ID behaviour
# We strictly DO NOT drop duplicates based on Task_ID.
# As established, multiple rows per Task_ID are expected because one task can have multiple timesheet log entries.

# Final verification
rows_dropped = initial_rows - df.shape[0]
print(f"\nTotal rows dropped in Step 5: {rows_dropped}")
print(f"Dataset shape after Step 5: {df.shape[0]} rows, {df.shape[1]} columns")

--- Step 5: Duplicate Handling ---
Found 3 completely duplicated rows. Dropping them...
Timesheet_ID was removed in Step 2; skipping Timesheet_ID duplicate check.

Total rows dropped in Step 5: 3
Dataset shape after Step 5: 11630 rows, 19 columns


In [12]:
print("--- Step 6: Cleaning Categorical Variables ---")

# 1. Define the categorical columns to target
categorical_cols = [
    'Employee_Department', 
    'Employee_Job_Position', 
    'Task_Priority', 
    'Task_Stage', 
    'Project_Name'
]

for col in categorical_cols:
    if col in df.columns:
        # 2. Standardize formatting: convert to string, strip whitespace, and apply Title Case
        df[col] = df[col].astype(str).str.strip().str.title()
        
        # 3. Catch any empty strings that might have been created after stripping spaces
        df[col] = df[col].replace({'': 'Unknown', 'Nan': 'Unknown', 'None': 'Unknown'})

# 4. Handle specific spelling inconsistencies and equivalent categories
# Note: You should check df['Project_Name'].unique() etc. to expand these dictionaries based on your specific dataset.
job_mapping = {
    'Dev': 'Developer',
    'Qa': 'Quality Assurance',
    'Project Mgr': 'Project Manager'
}

department_mapping = {
    'R&D': 'Research And Development (R&D)',
    'Research & Development': 'Research And Development (R&D)'
}

if 'Employee_Job_Position' in df.columns:
    df['Employee_Job_Position'] = df['Employee_Job_Position'].replace(job_mapping)

if 'Employee_Department' in df.columns:
    df['Employee_Department'] = df['Employee_Department'].replace(department_mapping)

# 5. Verify the unique categories to ensure the cleaning worked
for col in categorical_cols:
    if col in df.columns:
        print(f"\nUnique categories in '{col}' ({df[col].nunique()} total):")
        # Print the first 10 unique values as a quick sanity check
        print(df[col].unique()[:10])

--- Step 6: Cleaning Categorical Variables ---

Unique categories in 'Employee_Department' (8 total):
<ArrowStringArray>
['Research And Development (R&D)',                 'Colombo Branch',
              'Business Solution',          'Support & Maintenance',
              'Sales & Marketing',                        'Unknown',
         'Quality Assurance (Qa)',                 'Administration']
Length: 8, dtype: str

Unique categories in 'Employee_Job_Position' (20 total):
<ArrowStringArray>
['Team Lead - Research And Development (R&D)',
                            'Project Manager',
            'Associate Functional Consultant',
                'Associate Software Engineer',
    'Training Software Engineer - Internship',
                             'Support Intern',
        'Training Odoo Functional Consultant',
                          'Software Engineer',
               'Functional Support Executive',
                 'Odoo Functional Consultant']
Length: 10, dtype: str

Unique cat

In [13]:
print("--- Step 7: Cleaning Numerical Variables ---")

numerical_cols = [
    'Hours_Spent', 
    'Estimated_Planned_Hours', 
    'Actual_Hours_Spent', 
    'Timesheet_Logs_Count'
]

# 1. Handle Negative Values
# Time and counts cannot logically be negative. We convert any accidental negatives to absolute values.
for col in numerical_cols:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"Fixing {neg_count} negative values in '{col}' (converting to absolute value).")
            df[col] = df[col].abs()

# 2. Handle Unrealistic Values (Outliers)
# A single timesheet log entry ('Hours_Spent') logically cannot exceed 24 hours in a single day.
if 'Hours_Spent' in df.columns:
    unrealistic_hours = (df['Hours_Spent'] > 24).sum()
    if unrealistic_hours > 0:
        print(f"Capping {unrealistic_hours} unrealistic 'Hours_Spent' values at 24.0 hours.")
        df['Hours_Spent'] = df['Hours_Spent'].clip(upper=24.0)

# 3. Handle Zero Values
# As noted during EDA, zero values are logically valid for unestimated tasks and brief meetings. 
# We keep them exactly as they are.

# 4. Handle Missing Values / Incorrect Formats
# These were already addressed in Steps 3 and 4, but we run a quick sanity check.
missing_num = df[numerical_cols].isnull().sum().sum()
if missing_num > 0:
    print(f"Warning: Found {missing_num} missing numerical values. Filling with 0.0.")
    df[numerical_cols] = df[numerical_cols].fillna(0.0)

# 5. Verify the clean numerical distributions
print("\nNumerical Data Summary (Min, Max, Mean) after Cleaning:")
print(df[numerical_cols].describe().T[['min', 'max', 'mean']])

--- Step 7: Cleaning Numerical Variables ---

Numerical Data Summary (Min, Max, Mean) after Cleaning:
                         min      max        mean
Hours_Spent              0.0    24.00    3.613436
Estimated_Planned_Hours  0.0   552.00   10.156364
Actual_Hours_Spent       0.0  1070.87  131.141189
Timesheet_Logs_Count     1.0   441.00   50.163801


In [14]:
print("--- Step 8: Cleaning Date Columns ---")

date_cols = ['Date', 'Created_Date', 'Deadline_Date']

# 1. Ensure Date Formats (Sanity check following Step 4)
for col in date_cols:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Handle Missing Essential Dates
# 'Date' (when work was done) and 'Created_Date' (when task was made) are mandatory.
for col in ['Date', 'Created_Date']:
    if col in df.columns:
        missing_dates = df[col].isnull().sum()
        if missing_dates > 0:
            print(f"Dropping {missing_dates} rows due to missing essential '{col}'.")
            df.dropna(subset=[col], inplace=True)

# Note: We intentionally ignore missing values in 'Deadline_Date' as it is a valid business state.

# 3. Fix Chronological Errors (Deadline before Creation)
if 'Created_Date' in df.columns and 'Deadline_Date' in df.columns:
    # Find rows where the deadline is illogically set before the task was even created
    invalid_deadlines = df['Deadline_Date'] < df['Created_Date']
    invalid_count = invalid_deadlines.sum()
    
    if invalid_count > 0:
        print(f"Fixing {invalid_count} records where Deadline_Date is earlier than Created_Date.")
        # Safest approach: treat these invalid deadlines as missing data (NaT)
        df.loc[invalid_deadlines, 'Deadline_Date'] = pd.NaT

# 4. Verify Final State
print("\nDate Columns Status Summary:")
for col in date_cols:
    if col in df.columns:
        missing = df[col].isnull().sum()
        print(f" - {col}: {missing} missing values remaining.")

--- Step 8: Cleaning Date Columns ---
Fixing 143 records where Deadline_Date is earlier than Created_Date.

Date Columns Status Summary:
 - Date: 0 missing values remaining.
 - Created_Date: 0 missing values remaining.
 - Deadline_Date: 10153 missing values remaining.


In [15]:
import re

print("--- Step 9: Cleaning Text Columns ---")

# 1. Define the actual text columns currently in our dataset
text_cols = [
    'Task_Name', 
    'Work_Description', 
    'Task_Description', 
    'Timesheet_Work_Logs'
]

for col in text_cols:
    if col in df.columns:
        # 2. Convert missing to empty strings (reinforcing Step 3) and cast to string type
        df[col] = df[col].fillna("").astype(str)
        
        # 3. Convert to consistent case (lowercase is standard for NLP prep)
        df[col] = df[col].str.lower()
        
        # 4. Handle obvious formatting problems (replace newlines, carriage returns, and tabs with spaces)
        df[col] = df[col].str.replace(r'[\n\r\t]', ' ', regex=True)
        
        # 5. Normalize excessive whitespace (replace multiple spaces with a single space)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
        
        # 6. Remove leading and trailing whitespace
        df[col] = df[col].str.strip()

# 7. Verify the output
print("Text cleaning complete. Sample of cleaned 'Work_Description':")
if 'Work_Description' in df.columns:
    print(df['Work_Description'].head())

--- Step 9: Cleaning Text Columns ---
Text cleaning complete. Sample of cleaned 'Work_Description':
0                        db backup and restore in test
1                        db backup and restore in test
2                                           add addons
3                                                    /
4    get privillages to charith's new github accoun...
Name: Work_Description, dtype: str


In [16]:
print("--- Step 10: Validating Identifiers ---")

# 1. Validate Employee Consistency (Using df_raw for the dropped 'Employee_Name')
# We expect exactly 1 Name, 1 Department, and 1 Job Position per Employee_ID
emp_consistency = df_raw.groupby('Employee_ID').agg(
    Unique_Names=('Employee_Name', 'nunique'),
    Unique_Depts=('Employee_Department', 'nunique'),
    Unique_Jobs=('Employee_Job_Position', 'nunique')
)

# Filter for any Employee_ID that maps to more than one of these attributes
inconsistent_emps = emp_consistency[
    (emp_consistency['Unique_Names'] > 1) | 
    (emp_consistency['Unique_Depts'] > 1) | 
    (emp_consistency['Unique_Jobs'] > 1)
]

if not inconsistent_emps.empty:
    print(f"⚠️ Warning: Found {len(inconsistent_emps)} employees with inconsistent profile mappings!")
    print(inconsistent_emps)
else:
    print("✅ Employee_ID mappings (Name, Department, Job Position) are completely consistent.")

# 2. Validate Task -> Employee Relationships (Collaborations)
# Count how many unique Employee_IDs are associated with each Task_ID in our working dataset
task_collaborators = df.groupby('Task_ID')['Employee_ID'].nunique()
multi_employee_tasks = task_collaborators[task_collaborators > 1]

print(f"\n--- Task Assignment Verification ---")
print(f"Total unique tasks logged: {task_collaborators.count()}")
print(f"Tasks handled by a single employee: {sum(task_collaborators == 1)}")
print(f"Tasks handled by multiple employees (collaborations): {len(multi_employee_tasks)}")

if len(multi_employee_tasks) > 0:
    print(f"Maximum number of collaborators on a single task: {multi_employee_tasks.max()}")

--- Step 10: Validating Identifiers ---
✅ Employee_ID mappings (Name, Department, Job Position) are completely consistent.

--- Task Assignment Verification ---
Total unique tasks logged: 2442
Tasks handled by a single employee: 2000
Tasks handled by multiple employees (collaborations): 442
Maximum number of collaborators on a single task: 29


In [17]:
print("--- Step 11: Flagging Data Leakage Risks ---")

# 1. Define the columns that contain post-assignment information (leakage risks)
leakage_columns = [
    'Actual_Hours_Spent',          # Not known until the task is worked on.
    'Task_Stage',                  # Reflects the current/final state (e.g., 'Done').
    'Timesheet_Logs_Count',        # Calculated after timesheets are submitted.
    'Timesheet_Work_Logs',         # Work descriptions written after the work is performed.
    'All_Collaborating_Employees'  # Unknown until multiple people log time.
]

# 2. Add a "Red Flag" prefix to these columns instead of dropping them
rename_mapping = {col: f"FLAG_LEAKAGE_{col}" for col in leakage_columns if col in df.columns}
df.rename(columns=rename_mapping, inplace=True)

# 3. Verify the flagging
print("The following columns have been RED-FLAGGED for leakage and must be dropped before splitting/modelling:")
for old_name, new_name in rename_mapping.items():
    print(f" 🚩 {old_name}  --->  {new_name}")

print(f"\nDataset Shape remains: {df.shape[0]} rows, {df.shape[1]} columns")

--- Step 11: Flagging Data Leakage Risks ---
The following columns have been RED-FLAGGED for leakage and must be dropped before splitting/modelling:
 🚩 Actual_Hours_Spent  --->  FLAG_LEAKAGE_Actual_Hours_Spent
 🚩 Task_Stage  --->  FLAG_LEAKAGE_Task_Stage
 🚩 Timesheet_Logs_Count  --->  FLAG_LEAKAGE_Timesheet_Logs_Count
 🚩 Timesheet_Work_Logs  --->  FLAG_LEAKAGE_Timesheet_Work_Logs
 🚩 All_Collaborating_Employees  --->  FLAG_LEAKAGE_All_Collaborating_Employees

Dataset Shape remains: 11630 rows, 19 columns


In [18]:
print("==================================================")
print("     STEP 12: FINAL DATA QUALITY VALIDATION")
print("==================================================\n")

# 1. Compare Before vs After Shape
print("--- 1. Dataset Shape ---")
print(f"Before Preprocessing: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("          ↓")
print("     Preprocessing")
print("          ↓")
print(f"After Preprocessing:  {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# 2. Missing Value Summary
print("--- 2. Missing-Value Summary ---")
missing_after = df.isnull().sum()
missing_filtered = missing_after[missing_after > 0]
if missing_filtered.empty:
    print("✅ No missing values remaining (excluding valid NaT in dates).")
else:
    print("Expected missing values (e.g., open-ended Deadline_Dates):")
    print(missing_filtered.to_string())
print("\n")

# 3. Duplicate Summary
print("--- 3. Duplicate Summary ---")
print(f"Completely duplicated rows remaining: {df.duplicated().sum()} ✅\n")

# 4. Data Types
print("--- 4. Data Types ---")
print(df.dtypes.value_counts().to_string())
print("\n")

# 5. Unique Counts (Employees & Tasks)
print("--- 5. Unique Identifiers ---")
print(f"Unique Employee Count: {df['Employee_ID'].nunique()}")
print(f"Unique Task Count:     {df['Task_ID'].nunique()}\n")

# 6. Numerical Ranges
print("--- 6. Numerical Ranges (Sample) ---")
num_cols = ['Hours_Spent', 'Estimated_Planned_Hours']
for col in num_cols:
    if col in df.columns:
        print(f"{col}: Min = {df[col].min()}, Max = {df[col].max()}, Mean = {df[col].mean():.2f}")
print("\n")

# 7. Categorical Consistency
print("--- 7. Categorical Consistency ---")
cat_cols = ['Task_Priority', 'Employee_Department']
for col in cat_cols:
    if col in df.columns:
        print(f"{col} unique values: {df[col].nunique()} (Standardized to Title Case)")
print("\n")

# 8. Text Completeness
print("--- 8. Text Completeness ---")
text_cols = ['Task_Name', 'Work_Description']
for col in text_cols:
    if col in df.columns:
        empty_strings = (df[col] == "").sum()
        print(f"{col}: {empty_strings} empty strings remaining (Ready for NLP)")
print("\n")

# 9. ID Consistency (Check if any target values are null)
print("--- 9. ID Consistency ---")
print(f"Null Employee_IDs (Target): {df['Employee_ID'].isnull().sum()} ✅")
print(f"Null Task_IDs (Join Key):   {df['Task_ID'].isnull().sum()} ✅")

print("\n==================================================")
print("        PREPROCESSING COMPLETION SUMMARY")
print("==================================================")

     STEP 12: FINAL DATA QUALITY VALIDATION

--- 1. Dataset Shape ---
Before Preprocessing: 11,633 rows × 22 columns
          ↓
     Preprocessing
          ↓
After Preprocessing:  11,630 rows × 19 columns

--- 2. Missing-Value Summary ---
Expected missing values (e.g., open-ended Deadline_Dates):
Deadline_Date    10153


--- 3. Duplicate Summary ---
Completely duplicated rows remaining: 1 ✅

--- 4. Data Types ---
str               12
datetime64[us]     3
float64            3
int64              1


--- 5. Unique Identifiers ---
Unique Employee Count: 49
Unique Task Count:     2442

--- 6. Numerical Ranges (Sample) ---
Hours_Spent: Min = 0.0, Max = 24.0, Mean = 3.61
Estimated_Planned_Hours: Min = 0.0, Max = 552.0, Mean = 10.16


--- 7. Categorical Consistency ---
Task_Priority unique values: 2 (Standardized to Title Case)
Employee_Department unique values: 8 (Standardized to Title Case)


--- 8. Text Completeness ---
Task_Name: 0 empty strings remaining (Ready for NLP)
Work_Description

In [19]:
print("--- Preprocessed Dataset (First 5 Rows) ---")
display(df.head())

--- Preprocessed Dataset (First 5 Rows) ---


,Date,Task_ID,Task_Name,Work_Description,Hours_Spent,Project_Name,Employee_ID,Employee_Department,Employee_Job_Position,Task_Description,FLAG_LEAKAGE_Timesheet_Work_Logs,Task_Priority,Estimated_Planned_Hours,FLAG_LEAKAGE_Actual_Hours_Spent,FLAG_LEAKAGE_Timesheet_Logs_Count,FLAG_LEAKAGE_Task_Stage,Created_Date,Deadline_Date,FLAG_LEAKAGE_All_Collaborating_Employees
0,2026-09-14,TSK-1640,odoo sh maintain,db backup and restore in test,0.25,Hovael Project,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo sh maintain. work details: add addons t t...,add addons t the server | get backup from live...,Low,0.0,14.25,27,Project Preparation,2026-02-05,NaT,W M I L Wijesinghe
1,2026-09-14,TSK-1881,odoo sh maintain,db backup and restore in test,0.25,Ceylon Eco Spices,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo sh maintain. work details: add addon and ...,add addon and test | get odoo sh live ackup an...,Low,0.0,6.50,13,Developments,2026-03-19,NaT,W M I L Wijesinghe
2,2026-09-14,TSK-182,odoo.sh maintaing,add addons,0.25,Mihiri Bakemart (Pvt)Ltd - Development,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo.sh maintaing. work details: add all addon...,add all addon and build and test on the odoo s...,Low,0.0,40.75,29,Ongoing,2025-06-09,NaT,W M I L Wijesinghe
3,2026-09-14,TSK-2884,development meeting,/,0.00,Cygnus One,EMP-55,Colombo Branch,Project Manager,development meeting. work details: intern deve...,intern development hoveal project meeting | me...,Low,0.0,9.42,13,Miscellaneous,2026-07-06,NaT,"W M I L Wijesinghe, K R V Dias, A R M S Madusa..."
4,2026-09-14,TSK-405,other tasks (mention on description),get privillages to charith's new github accoun...,0.50,Cygnus One,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),other tasks (mention on description). work det...,self ssl setup on vps | preparing report list ...,Low,0.0,162.18,69,Miscellaneous,2025-07-14,NaT,"H.M.C.S Thilakarathna, Sadaruwan Bandara, L H ..."


In [20]:
import os

print("--- Step 13: Saving the Preprocessed Dataset ---")

# 1. Define the file path for the processed data folder
# Using the relative path assuming this is run from the 'notebooks/' directory
output_dir = '../data/processed'
output_file = 'preprocessed_employee_task_data.csv'
output_path = os.path.join(output_dir, output_file)

# 2. Save the cleaned DataFrame to CSV
# Setting index=False ensures Pandas doesn't write the row numbers as a new column
df.to_csv(output_path, index=False)

print(f"✅ Preprocessed dataset successfully saved to: {output_path}")
print("Your data is officially ready for the Feature Engineering notebook!")

--- Step 13: Saving the Preprocessed Dataset ---
✅ Preprocessed dataset successfully saved to: ../data/processed/preprocessed_employee_task_data.csv
Your data is officially ready for the Feature Engineering notebook!
